## **Import & Setup**

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys
import numpy as np
import pandas as pd
from scipy.spatial import Delaunay
from scipy.stats import kendalltau
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

sys.path.append("../src")
from evaluation import evaluate, print_results

## **Help Functions**

In [11]:
# Function that partitions neighborhoods into N contiguous, size-balanced macro-zones
def random_contiguous_partition(n_zones, seed):
    random_generator = np.random.default_rng(seed)
    number_of_neighbourhoods = len(neighbourhoods)
    seed_neighbourhoods = random_generator.choice(number_of_neighbourhoods, size=n_zones, replace=False)
    zone_of = np.full(number_of_neighbourhoods, -1)
    for zone_index, neighbourhood in enumerate(seed_neighbourhoods):
        zone_of[neighbourhood] = zone_index
    zone_sizes = [1] * n_zones
    unassigned = set(range(number_of_neighbourhoods)) - set(seed_neighbourhoods.tolist())
    while unassigned:
        growable_zones = []
        for zone_index in range(n_zones):
            members = np.where(zone_of == zone_index)[0]
            available_neighbours = set().union(*[adjacency[member] for member in members]) & unassigned
            if available_neighbours:
                growable_zones.append((zone_index, list(available_neighbours)))
        if not growable_zones:
            for stranded in list(unassigned):
                assigned = np.where(zone_of >= 0)[0]
                nearest = assigned[np.linalg.norm(centroids[assigned] - centroids[stranded], axis=1).argmin()]
                zone_of[stranded] = zone_of[nearest]
                unassigned.discard(stranded)
            break
        growth_weights = np.array([1.0 / zone_sizes[zone_index] for zone_index, _ in growable_zones])
        growth_weights = growth_weights / growth_weights.sum()
        chosen_zone, candidate_neighbours = growable_zones[random_generator.choice(len(growable_zones), p=growth_weights)]
        new_member = random_generator.choice(candidate_neighbours)
        zone_of[new_member] = chosen_zone
        zone_sizes[chosen_zone] += 1
        unassigned.discard(new_member)
    return zone_of

In [12]:
# score-function builders (same formulations as the baselines) 
def make_toppop_score_fn(train_matrix):
    popularity = train_matrix.sum(axis=0)
    def score_fn(nace):
        return popularity
    return score_fn

def make_itemcf_score_fn(train_matrix):
    similarity = pd.DataFrame(
        cosine_similarity(train_matrix.T.values),
        index=train_matrix.columns,
        columns=train_matrix.columns,
    )
    def score_fn(nace):
        return pd.Series(similarity.values @ train_matrix.loc[nace].values, index=train_matrix.columns)
    return score_fn

def make_puresvd_score_fn(train_matrix, n_factors):
    n_components = min(n_factors, min(train_matrix.shape) - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    svd.fit(train_matrix.values)
    item_factors = svd.components_.T
    reconstruction = item_factors @ item_factors.T
    def score_fn(nace):
        return pd.Series(train_matrix.loc[nace].values @ reconstruction, index=train_matrix.columns)
    return score_fn

def make_random_score_fn(train_matrix, seed=0):
    random_generator = np.random.default_rng(seed)
    columns = train_matrix.columns
    def score_fn(nace):
        return pd.Series(random_generator.random(len(columns)), index=columns)
    return score_fn

In [13]:
# Function that evaluates all the methods (PureSVD gets fine-tuned first)
def evaluate_all_methods(train_matrix, evaluation_full_matrix, test_pairs, validation_pairs, head, svd_grid):
    summaries = {}
    summaries["TopPop"], _ = evaluate(make_toppop_score_fn(train_matrix), test_pairs, evaluation_full_matrix, head_neighborhoods=head)
    summaries["ItemCF"], _ = evaluate(make_itemcf_score_fn(train_matrix), test_pairs, evaluation_full_matrix, head_neighborhoods=head)
    summaries["Random"], _ = evaluate(make_random_score_fn(train_matrix), test_pairs, evaluation_full_matrix, head_neighborhoods=head)
    
    best_factors = None
    best_validation_score = -1
    for n_factors in svd_grid:
        validation_summary, _ = evaluate(make_puresvd_score_fn(train_matrix, n_factors), validation_pairs, evaluation_full_matrix, head_neighborhoods=head)
        if validation_summary["full"]["NDCG@10"] > best_validation_score:
            best_validation_score = validation_summary["full"]["NDCG@10"]
            best_factors = n_factors
    summaries["PureSVD"], _ = evaluate(make_puresvd_score_fn(train_matrix, best_factors), test_pairs, evaluation_full_matrix, head_neighborhoods=head)
    return summaries

In [ ]:
# Returns the ordered results of either the full test or the long tail
def method_ordering(summaries, subset):
    return sorted(summaries.keys(), key=lambda method: -summaries[method][subset]["NDCG@10"])

In [ ]:
def build_zone_problem(zone_of, n_zones, split_seed, nace_codes):
    zone_ids = [f"zone_{zone_index}" for zone_index in range(n_zones)]
    zone_counts = np.zeros((len(nace_codes), n_zones))
    for neighbourhood in range(len(neighbourhoods)):
        zone_counts[:, zone_of[neighbourhood]] += full_values[:, neighbourhood]
    zone_full_matrix = pd.DataFrame(zone_counts, index=nace_codes, columns=zone_ids)

    is_present = zone_full_matrix.values > 0
    train_values = zone_full_matrix.values.copy()
    test_rows = []
    validation_rows = []
    random_generator = np.random.default_rng(split_seed)
    for row_index, nace in enumerate(nace_codes):
        present_zones = np.where(is_present[row_index])[0]
        if len(present_zones) >= 3:
            held_out = random_generator.choice(present_zones, size=2, replace=False)
            train_values[row_index, held_out[0]] = 0
            train_values[row_index, held_out[1]] = 0
            test_rows.append({"nace": nace, "neighborhood": zone_ids[held_out[0]]})
            validation_rows.append({"nace": nace, "neighborhood": zone_ids[held_out[1]]})
        elif len(present_zones) == 2:
            held_out = random_generator.choice(present_zones)
            train_values[row_index, held_out] = 0
            test_rows.append({"nace": nace, "neighborhood": zone_ids[held_out]})
    zone_train_matrix = pd.DataFrame(train_values, index=nace_codes, columns=zone_ids)
    head_size = max(1, round(0.21 * n_zones))
    head_zones = set(zone_train_matrix.sum(axis=0).sort_values(ascending=False).head(head_size).index)
    return zone_full_matrix, zone_train_matrix, pd.DataFrame(test_rows), pd.DataFrame(validation_rows), head_zones

## **Load the Data**

In [9]:
DATA = "../data"
full_matrix         = pd.read_parquet(f"{DATA}/interaction_matrix_raw.parquet")
features            = pd.read_parquet(f"{DATA}/neighborhood_features.parquet")
train_matrix_nbhd   = pd.read_parquet(f"{DATA}/train_matrix.parquet")
test_pairs_nbhd     = pd.read_parquet(f"{DATA}/test_pairs.parquet")
val_pairs_nbhd      = pd.read_parquet(f"{DATA}/val_pairs.parquet")

neighbourhoods   = list(full_matrix.columns)
nace_codes       = list(full_matrix.index)
centroids        = features.loc[neighbourhoods, ["Centroid_x", "Centroid_y"]].values
full_values      = full_matrix.values

In [ ]:
triangulation = Delaunay(centroids)
adjacency = {index: set() for index in range(len(neighbourhoods))}
for simplex in triangulation.simplices:
    for first in simplex:
        for second in simplex:
            if first != second:
                adjacency[first].add(second)
print("average adjacency degree:", round(np.mean([len(neighbours) for neighbours in adjacency.values()]), 1))

average adjacency degree: 5.6


## **Experiment with Original Spatial Units**

In [ ]:
neighbourhood_head = set(train_matrix_nbhd.sum(axis=0).sort_values(ascending=False).head(10).index)
reference_summaries = evaluate_all_methods(train_matrix_nbhd, full_matrix, test_pairs_nbhd, val_pairs_nbhd, neighbourhood_head, [10, 20, 30, 40])
reference_full_order = method_ordering(reference_summaries, "full")
reference_tail_order = method_ordering(reference_summaries, "long_tail")

print("reference full-test order:", reference_full_order)
print("reference long-tail order:", reference_tail_order)

reference full-test order: ['TopPop', 'ItemCF', 'PureSVD', 'Random']
reference long-tail order: ['PureSVD', 'ItemCF', 'TopPop', 'Random']


We see that we re-produce the order of the original baselines notebook using the original spatial unit (48 neighborhoods)

## **Experiment with 50 + 50 Random Spatial Units**

We use 2 scales: N = 24 and N = 16 spatial units. For each scale we fine-tune PureSVD, and compare the results with the original spatial unit above

In [ ]:
# MAUP robustness across scales (50 random contiguous re-zonings each)
# We keep multiple stats to see the difference (if TopPop collapses in long tail / if the order preserves across different settings/...)
methods = ["TopPop", "ItemCF", "PureSVD", "Random"]
number_of_trials = 50
scale_settings = [(24, [4, 8, 12, 16, 20]), (16, [4, 8, 12])]

for n_zones, svd_grid in scale_settings:
    full_taus = []
    tail_taus = []
    full_order_preserved = 0
    longtail_collapse_replicated = 0
    itemcf_beats_toppop_on_tail = 0

    # Repeat expriment 50 times:
    for trial in range(number_of_trials):

        # extract spatial units after random re-zoning
        zone_of = random_contiguous_partition(n_zones, seed=3000 + trial)
        zone_full, zone_train, test_pairs_zone, val_pairs_zone, head_zones = build_zone_problem(zone_of, n_zones, split_seed=7000 + trial, nace_codes=nace_codes)
        validation_for_tuning = val_pairs_zone if len(val_pairs_zone) else test_pairs_zone
        
        # evaluate the results
        summaries = evaluate_all_methods(zone_train, zone_full, test_pairs_zone, validation_for_tuning, head_zones, svd_grid)

        # update statistics based on results
        full_order = method_ordering(summaries, "full")
        tail_order = method_ordering(summaries, "long_tail")
        full_taus.append(kendalltau([reference_full_order.index(m) for m in methods], [full_order.index(m) for m in methods]).correlation)
        tail_taus.append(kendalltau([reference_tail_order.index(m) for m in methods], [tail_order.index(m) for m in methods]).correlation)
        full_order_preserved += (full_order == reference_full_order)
        longtail_collapse_replicated += (summaries["PureSVD"]["long_tail"]["NDCG@10"] > summaries["TopPop"]["long_tail"]["NDCG@10"])
        itemcf_beats_toppop_on_tail += (summaries["ItemCF"]["long_tail"]["NDCG@10"] > summaries["TopPop"]["long_tail"]["NDCG@10"])
    
    print(f"N={n_zones}: "
          f"full-tau {np.nanmean(full_taus):.2f} +/- {np.nanstd(full_taus):.2f} | "
          f"tail-tau {np.nanmean(tail_taus):.2f} +/- {np.nanstd(tail_taus):.2f} | "
          f"exact full order {full_order_preserved}/{number_of_trials} | "
          f"long-tail collapse {longtail_collapse_replicated}/{number_of_trials} | "
          f"ItemCF>TopPop tail {itemcf_beats_toppop_on_tail}/{number_of_trials}"
    )

N=24: full-tau 0.99 +/- 0.05 | tail-tau 0.69 +/- 0.26 | exact full order 49/50 | long-tail collapse 41/50 | ItemCF>TopPop tail 27/50
N=16: full-tau 0.97 +/- 0.10 | tail-tau 0.31 +/- 0.33 | exact full order 45/50 | long-tail collapse 14/50 | ItemCF>TopPop tail 22/50
